In [ ]:
# import os
# import sys
# project_path = os.path.abspath(os.path.join(os.getcwd(), "../../"))
# sys.path.append(project_path)
#
# requirements_path = os.path.join(project_path, "SECONDARY/requirements.txt")
# !{sys.executable} -m pip install -r "{requirements_path}"

In [ ]:
import os
import sys
import time
!{sys.executable} --version
if sys.version_info.minor == 8:
    raise RuntimeError('USE JUPYTER KERNEL VENV 3.10/310/DEFAULT INSTEAD')

!cd /workspace/CRYPTO_MACAQUES && pip install .
!cd /home/crypto/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && source venv/bin/activate && !{sys.executable} -m pip install .

from IPython.core.display_functions import clear_output
clear_output()

In [ ]:
%load_ext autoreload
%autoreload

from datetime import timedelta
from dateutil import parser
from SRC.LIBRARIES.new_data_utils import fetch
from SRC.CORE._CONSTANTS import _KIEV_TIMESTAMP
import SRC.LIBRARIES.new_utils as nu

# todo xrp 4h 2025 and 2026 champion.
# todo doge/xrp 4h 2026 champions
sym='xrp'.upper()
symbol = f'{sym}USDT'
discretization = '4h'.upper()
display_start_date_str = '2026-01-01'
display_end_date_str = '2026-05-21'
load_end_date = parser.parse(display_end_date_str)
capital_per_trade = 1000
commission_rate = 0.00075

market_type = 'SPOT'
mrc_buffer = 1000
rsi_buffer = 200
stoch_buffer = 50
macd_buffer = 200
atr_buffer = 200
display_start_dt = parser.parse(display_start_date_str)
start_time = time.perf_counter()

discretization_value = int(discretization[:-1])
timeframe_seconds = {
    'M': discretization_value * 60,
    'H': discretization_value * 3600,
    'D': discretization_value * 86400
}.get(discretization[-1], 1800)

load_start_dt = display_start_dt - timedelta(seconds=timeframe_seconds * mrc_buffer)

mrc_df = fetch(market_type, symbol, discretization, load_start_dt, load_end_date)
df = mrc_df.iloc[mrc_buffer:].copy()

In [ ]:
# Убеждаемся, что индексы уникальны
mrc_df = mrc_df[~mrc_df.index.duplicated(keep='first')]
df = df[~df.index.duplicated(keep='first')]

# 1. RSI
rsi_calc_df = nu.prepare_buffer_data(mrc_df, df, rsi_buffer)
df = nu.rsi(df, rsi_calc_df)

# 2. Stochastic
stoch_calc_df = nu.prepare_buffer_data(mrc_df, df, stoch_buffer)
df = nu.stochastic_tradingview(df, stoch_calc_df)

# 3. MACD
macd_calc_df = nu.prepare_buffer_data(mrc_df, df, macd_buffer)
df, macd = nu.macd(df, macd_calc_df)

# 4. ATR
atr_calc_df = nu.prepare_buffer_data(mrc_df, df, atr_buffer)
df = nu.atr(df, atr_calc_df)

# Расчёт MRC
mrc_df = nu.mrc_calculate(mrc_df, df)

# SMA для фильтра (период можно менять)
sma_period = 20
df['sma'] = df['close'].rolling(sma_period).mean()

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, Tuple

def backtest_advanced(df: pd.DataFrame,
                       commission: float = 0.00075,
                       capital_per_trade: float = 1000.0,
                       slippage: float = 0.0002,
                       lookback_min_bars: int = 8,
                       slope_threshold: float = 0.0,
                       use_slope_filter: bool = True,
                       use_stoch_filter: bool = True,
                       stoch_oversold: float = 30.0,
                       stoch_overbought: float = 70.0,
                       entry_mode: str = 'green_touch',
                       exit_mode: str = 'opposite_green',
                       exit_mode_for_green: str = 'opposite_green',
                       trailing_tp: bool = False,
                       trailing_tp_distance: float = 0.5,
                       partial_first_percent: float = 50.0,
                       breakout_confirmation: int = 1,
                       yellow_sl_mode: str = 'green',
                       use_yellow_entry: bool = True
                      ) -> Tuple[pd.DataFrame, Dict]:
    """
    Расширенная стратегия MRC + Стохастик.
    
    entry_mode:
      - 'green_touch': только касание зелёной линии
      - 'green_breakout': закрытие выше/ниже зелёной линии
      - 'yellow_touch': только касание жёлтой линии
      - 'all': все варианты (зависит от use_yellow_entry)

    exit_mode:
      - 'opposite_green': противоположная зелёная линия
      - 'yellow': жёлтая линия
      - 'partial': частичная фиксация (первый TP на жёлтой)
      
    exit_mode_for_green:
      - 'opposite_green': для сделок с зелёной линии выход на противоположной зелёной
      - 'yellow': для сделок с зелёной линии выход на жёлтой линии (фиксируется при входе)
      
    yellow_sl_mode:
      - 'green': стоп на противоположной зелёной линии
      - 'red': стоп на красной линии
      - 'ratio': стоп с фиксированным соотношением риск/прибыль (1:3)
      
    use_yellow_entry:
      - True: разрешить вход от жёлтой линии
      - False: только зелёная линия
    """
    df_back = df.copy()

    # Наклон meanline
    df_back['slope'] = df_back['meanline'] - df_back['meanline'].shift(1)
    df_back['slope_dir'] = 'neutral'
    df_back.loc[df_back['slope'] > slope_threshold, 'slope_dir'] = 'up'
    df_back.loc[df_back['slope'] < -slope_threshold, 'slope_dir'] = 'down'

    # Зоны стохастика
    stoch_long_zone = df_back['stoch_k'] < stoch_oversold
    stoch_short_zone = df_back['stoch_k'] > stoch_overbought

    # ========== УСЛОВИЯ ВХОДА ==========
    if entry_mode == 'green_touch':
        long_touch = (df_back['low'] <= df_back['loband1']) & (df_back['high'] >= df_back['loband1'])
        short_touch = (df_back['high'] >= df_back['upband1']) & (df_back['low'] <= df_back['upband1'])
        long_base = long_touch
        short_base = short_touch

    elif entry_mode == 'green_breakout':
        long_base = df_back['close'] > df_back['upband1'].shift(breakout_confirmation)
        short_base = df_back['close'] < df_back['loband1'].shift(breakout_confirmation)

    elif entry_mode == 'yellow_touch':
        if use_yellow_entry:
            long_touch = (df_back['low'] <= df_back['meanline']) & (df_back['high'] >= df_back['meanline'])
            short_touch = (df_back['high'] >= df_back['meanline']) & (df_back['low'] <= df_back['meanline'])
            long_base = long_touch
            short_base = short_touch
        else:
            long_base = pd.Series(False, index=df_back.index)
            short_base = pd.Series(False, index=df_back.index)

    else:  # 'all' - комбинация
        long_touch1 = (df_back['low'] <= df_back['loband1']) & (df_back['high'] >= df_back['loband1'])
        long_base = long_touch1
        short_touch1 = (df_back['high'] >= df_back['upband1']) & (df_back['low'] <= df_back['upband1'])
        short_base = short_touch1
        
        if use_yellow_entry:
            long_touch2 = (df_back['low'] <= df_back['meanline']) & (df_back['high'] >= df_back['meanline'])
            short_touch2 = (df_back['high'] >= df_back['meanline']) & (df_back['low'] <= df_back['meanline'])
            long_base = long_base | long_touch2
            short_base = short_base | short_touch2

    # Фильтр по стохастику
    if use_stoch_filter:
        long_base = long_base & stoch_long_zone
        short_base = short_base & stoch_short_zone

    # Фильтр по наклону
    if use_slope_filter:
        long_base = long_base & (df_back['slope_dir'] == 'up')
        short_base = short_base & (df_back['slope_dir'] == 'down')

    # Инициализация
    df_back['signal'] = 0
    df_back['entry_price'] = np.nan
    df_back['exit_price'] = np.nan
    df_back['pnl'] = 0.0
    df_back['trade_type'] = ''

    position = None
    trades = []

    for i in range(1, len(df_back)):
        current_long_signal = long_base.iloc[i]
        current_short_signal = short_base.iloc[i]

        # Вход
        if position is None:
            can_enter = True
            if len(trades) > 0 and (i - trades[-1]['exit_idx']) < lookback_min_bars:
                can_enter = False

            if can_enter:
                if current_long_signal:
                    # Определяем, от какой линии пришёл сигнал
                    if use_yellow_entry and entry_mode == 'all' and 'long_touch2' in locals() and long_touch2.iloc[i] and not long_touch1.iloc[i]:
                        is_yellow_entry = True
                    else:
                        is_yellow_entry = False

                    # Цена входа
                    if is_yellow_entry:
                        raw_entry = df_back.iloc[i]['meanline']
                    else:
                        raw_entry = df_back.iloc[i]['loband1']
                    
                    entry_price = raw_entry * (1 + slippage)
                    amount = (capital_per_trade * (1 - commission)) / entry_price
                    
                    # TP - фиксируется при входе (не динамический)
                    if is_yellow_entry:
                        # Для жёлтых входов TP всегда на противоположной зелёной
                        tp_price = df_back.iloc[i]['upband1']
                    else:
                        # Для зелёных входов TP зависит от exit_mode_for_green
                        if exit_mode_for_green == 'yellow':
                            tp_price = df_back.iloc[i]['meanline']  # фиксируем жёлтую линию
                        else:
                            tp_price = df_back.iloc[i]['upband1']
                    
                    # SL в зависимости от типа входа и режима
                    if is_yellow_entry:
                        if yellow_sl_mode == 'green':
                            sl_price = df_back.iloc[i]['loband1']
                        elif yellow_sl_mode == 'red':
                            sl_price = df_back.iloc[i]['loband2']
                        else:  # 'ratio'
                            distance_to_tp = tp_price - entry_price
                            sl_price = entry_price - distance_to_tp / 3
                    else:
                        sl_price = df_back.iloc[i]['loband2']

                    position = {
                        'type': 'long',
                        'entry_idx': i,
                        'entry_price': entry_price,
                        'amount': amount,
                        'tp': tp_price,
                        'initial_tp': tp_price,
                        'sl': sl_price,
                        'initial_sl': sl_price,
                        'is_yellow_entry': is_yellow_entry,
                        'tp_trailing_active': trailing_tp,
                        'tp_highest_price': entry_price,
                        'partial_closed': False,
                        'partial_amount': amount * (partial_first_percent / 100) if exit_mode == 'partial' else 0
                    }
                    df_back.at[df_back.index[i], 'signal'] = 1
                    df_back.at[df_back.index[i], 'entry_price'] = entry_price
                    continue

                if current_short_signal:
                    # Определяем, от какой линии пришёл сигнал
                    if use_yellow_entry and entry_mode == 'all' and 'short_touch2' in locals() and short_touch2.iloc[i] and not short_touch1.iloc[i]:
                        is_yellow_entry = True
                    else:
                        is_yellow_entry = False

                    # Цена входа
                    if is_yellow_entry:
                        raw_entry = df_back.iloc[i]['meanline']
                    else:
                        raw_entry = df_back.iloc[i]['upband1']
                    
                    entry_price = raw_entry * (1 - slippage)
                    amount = (capital_per_trade * (1 - commission)) / entry_price
                    
                    # TP - фиксируется при входе (не динамический)
                    if is_yellow_entry:
                        # Для жёлтых входов TP всегда на противоположной зелёной
                        tp_price = df_back.iloc[i]['loband1']
                    else:
                        # Для зелёных входов TP зависит от exit_mode_for_green
                        if exit_mode_for_green == 'yellow':
                            tp_price = df_back.iloc[i]['meanline']  # фиксируем жёлтую линию
                        else:
                            tp_price = df_back.iloc[i]['loband1']
                    
                    # SL в зависимости от типа входа и режима
                    if is_yellow_entry:
                        if yellow_sl_mode == 'green':
                            sl_price = df_back.iloc[i]['upband1']
                        elif yellow_sl_mode == 'red':
                            sl_price = df_back.iloc[i]['upband2']
                        else:  # 'ratio'
                            distance_to_tp = entry_price - tp_price
                            sl_price = entry_price + distance_to_tp / 3
                    else:
                        sl_price = df_back.iloc[i]['upband2']

                    position = {
                        'type': 'short',
                        'entry_idx': i,
                        'entry_price': entry_price,
                        'amount': amount,
                        'tp': tp_price,
                        'initial_tp': tp_price,
                        'sl': sl_price,
                        'initial_sl': sl_price,
                        'is_yellow_entry': is_yellow_entry,
                        'tp_trailing_active': trailing_tp,
                        'tp_lowest_price': entry_price,
                        'partial_closed': False,
                        'partial_amount': amount * (partial_first_percent / 100) if exit_mode == 'partial' else 0
                    }
                    df_back.at[df_back.index[i], 'signal'] = -1
                    df_back.at[df_back.index[i], 'entry_price'] = entry_price
                    continue

        # Выход
        else:
            current_high = df_back.iloc[i]['high']
            current_low = df_back.iloc[i]['low']
            exit_price = None
            exit_type = None

            if position['type'] == 'long':
                # Обновление максимума для трейлинга
                if current_high > position['tp_highest_price']:
                    position['tp_highest_price'] = current_high

                # Трейлинг-ТП
                if position['tp_trailing_active']:
                    new_tp = position['tp_highest_price'] * (1 - trailing_tp_distance / 100)
                    if new_tp > position['initial_tp']:
                        position['tp'] = max(position['tp'], new_tp)

                # Проверка выходов
                if current_high >= position['tp']:
                    exit_price = position['tp']
                    exit_type = 'tp_trailing' if position['tp_trailing_active'] else 'tp'
                elif current_low <= position['sl']:
                    exit_price = position['sl']
                    exit_type = 'sl'

            else:  # short
                # Обновление минимума для трейлинга
                if current_low < position['tp_lowest_price']:
                    position['tp_lowest_price'] = current_low

                # Трейлинг-ТП
                if position['tp_trailing_active']:
                    new_tp = position['tp_lowest_price'] * (1 + trailing_tp_distance / 100)
                    if new_tp < position['initial_tp']:
                        position['tp'] = min(position['tp'], new_tp)

                # Проверка выходов
                if current_low <= position['tp']:
                    exit_price = position['tp']
                    exit_type = 'tp_trailing' if position['tp_trailing_active'] else 'tp'
                elif current_high >= position['sl']:
                    exit_price = position['sl']
                    exit_type = 'sl'

            if exit_price is not None:
                if position['type'] == 'long':
                    exit_price_adj = exit_price * (1 - slippage)
                    exit_proceeds = position['amount'] * exit_price_adj * (1 - commission)
                    pnl = exit_proceeds - capital_per_trade
                else:
                    exit_price_adj = exit_price * (1 + slippage)
                    entry_proceeds = position['amount'] * position['entry_price'] * (1 - commission)
                    exit_cost = position['amount'] * exit_price_adj * (1 + commission)
                    pnl = entry_proceeds - exit_cost

                trades.append({
                    'entry_idx': position['entry_idx'],
                    'exit_idx': i,
                    'type': position['type'],
                    'entry_price': position['entry_price'],
                    'exit_price': exit_price_adj,
                    'pnl': pnl,
                    'exit_type': exit_type
                })
                df_back.at[df_back.index[i], 'exit_price'] = exit_price_adj
                df_back.at[df_back.index[i], 'pnl'] = pnl
                df_back.at[df_back.index[i], 'trade_type'] = f"{position['type']}_{exit_type}"
                position = None

    # Принудительное закрытие
    if position is not None:
        last_price = df_back.iloc[-1]['close']
        if position['type'] == 'long':
            exit_price_adj = last_price * (1 - slippage)
            exit_proceeds = position['amount'] * exit_price_adj * (1 - commission)
            pnl = exit_proceeds - capital_per_trade
        else:
            exit_price_adj = last_price * (1 + slippage)
            entry_proceeds = position['amount'] * position['entry_price'] * (1 - commission)
            exit_cost = position['amount'] * exit_price_adj * (1 + commission)
            pnl = entry_proceeds - exit_cost
        trades.append({
            'entry_idx': position['entry_idx'],
            'exit_idx': len(df_back)-1,
            'type': position['type'],
            'entry_price': position['entry_price'],
            'exit_price': exit_price_adj,
            'pnl': pnl,
            'exit_type': 'forced'
        })
        df_back.at[df_back.index[-1], 'exit_price'] = exit_price_adj
        df_back.at[df_back.index[-1], 'pnl'] = pnl
        df_back.at[df_back.index[-1], 'trade_type'] = f"{position['type']}_forced"

    # Метрики
    trades_df = pd.DataFrame(trades)
    if len(trades_df) == 0:
        print("Нет сделок.")
        return df_back, {'total_pnl': 0, 'avg_return_per_trade_percent': 0, 'win_rate': 0, 'num_trades': 0, 'avg_win': 0, 'avg_loss': 0, 'max_drawdown': 0, 'sharpe_approx': 0}

    total_pnl = trades_df['pnl'].sum()
    avg_return_per_trade_percent = (total_pnl / (capital_per_trade * len(trades_df))) * 100
    win_rate = (trades_df['pnl'] > 0).mean()
    num_trades = len(trades_df)
    avg_win = trades_df[trades_df['pnl'] > 0]['pnl'].mean() if (trades_df['pnl'] > 0).any() else 0.0
    avg_loss = trades_df[trades_df['pnl'] < 0]['pnl'].mean() if (trades_df['pnl'] < 0).any() else 0.0
    cum_pnl = trades_df['pnl'].cumsum()
    running_max = cum_pnl.cummax()
    max_drawdown = (running_max - cum_pnl).max()
    returns = trades_df['pnl'] / capital_per_trade
    sharpe = (returns.mean() / returns.std()) * np.sqrt(252) if len(returns) > 1 and returns.std() != 0 else 0.0

    metrics = {
        'total_pnl': total_pnl,
        'avg_return_per_trade_percent': avg_return_per_trade_percent,
        'win_rate': win_rate,
        'num_trades': num_trades,
        'avg_win': avg_win,
        'avg_loss': avg_loss,
        'max_drawdown': max_drawdown,
        'sharpe_approx': sharpe
    }
    return df_back, metrics

In [ ]:
%load_ext autoreload
%autoreload

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import SRC.LIBRARIES.new_indicator_plot_utils as nipu

# ========== НАСТРОЙКИ (меняйте здесь для тестов) ==========
# Основные фильтры
USE_STOCH_FILTER = True
USE_SLOPE_FILTER = True

# Параметры стохастика
STOCH_OVERSOLD = 30.0
STOCH_OVERBOUGHT = 70.0

# ========== РЕЖИМЫ ВХОДА ==========
# 'green_touch'    - только касание зелёной линии
# 'green_breakout' - закрытие выше/ниже зелёной линии
# 'yellow_touch'   - только касание жёлтой линии
# 'all'            - комбинация (зелёная + опционально жёлтая)
ENTRY_MODE = 'all'

# ========== РАЗРЕШИТЬ ВХОД НА ЖЁЛТОЙ ЛИНИИ ==========
# True  - разрешить входы от жёлтой линии
# False - только зелёная линия
USE_YELLOW_ENTRY = False

# ========== РЕЖИМЫ ВЫХОДА ==========
# 'opposite_green' - противоположная зелёная линия
# 'yellow'         - жёлтая линия
# 'partial'        - частичная фиксация (50% на жёлтой, 50% на зелёной)
EXIT_MODE = 'opposite_green'

# ========== ВЫХОД ДЛЯ СДЕЛОК С ЗЕЛЁНОЙ ЛИНИИ ==========
# 'opposite_green' - противоположная зелёная линия
# 'yellow'         - жёлтая линия (фиксируется при входе)
EXIT_MODE_FOR_GREEN = 'opposite_green'

# ========== НАСТРОЙКИ СТОПА ДЛЯ ВХОДА НА ЖЁЛТОЙ ==========
# 'green' - стоп на противоположной зелёной (ближний)
# 'red'   - стоп на красной (дальний)
# 'ratio' - фиксированное соотношение 1:3
YELLOW_SL_MODE = 'red'

# ========== ТРЕЙЛИНГ ==========
TRAILING_TP = True
TRAILING_TP_DISTANCE = 0.5

# Параметры для partial
PARTIAL_FIRST_PERCENT = 50.0

# ==================================

is_log_scale_by_default = True
candlestick_row = 1

fig_main = make_subplots(rows=1, cols=1, shared_xaxes=True, vertical_spacing=0.03)

fig_main.add_trace(
    go.Candlestick(
        x=df[_KIEV_TIMESTAMP],
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        name="OHLC"
    ),
    row=candlestick_row, col=1
)

nipu.add_mrc(candlestick_row, fig_main, df)

# Запуск бэктеста
df_backtest, metrics = backtest_advanced(
    df,
    commission=commission_rate,
    capital_per_trade=capital_per_trade,
    slippage=0.0002,
    lookback_min_bars=8,
    slope_threshold=0.0,
    use_slope_filter=USE_SLOPE_FILTER,
    use_stoch_filter=USE_STOCH_FILTER,
    stoch_oversold=STOCH_OVERSOLD,
    stoch_overbought=STOCH_OVERBOUGHT,
    entry_mode=ENTRY_MODE,
    exit_mode=EXIT_MODE,
    exit_mode_for_green=EXIT_MODE_FOR_GREEN,
    trailing_tp=TRAILING_TP,
    trailing_tp_distance=TRAILING_TP_DISTANCE,
    partial_first_percent=PARTIAL_FIRST_PERCENT,
    yellow_sl_mode=YELLOW_SL_MODE,
    use_yellow_entry=USE_YELLOW_ENTRY
)

print(f"Результаты расширенной стратегии MRC+Stoch:")
print(f"{sym} {discretization} | {display_start_date_str} - {display_end_date_str}")
print(f"Режим входа: {ENTRY_MODE}")
print(f"Вход на жёлтой: {USE_YELLOW_ENTRY}")
print(f"Режим выхода: {EXIT_MODE}")
print(f"Выход для зелёных сделок: {EXIT_MODE_FOR_GREEN}")
print(f"Режим SL для жёлтой линии: {YELLOW_SL_MODE}")
print(f"Фильтры: Stoch={USE_STOCH_FILTER}, Slope={USE_SLOPE_FILTER}")
print(f"Стохастик: oversold={STOCH_OVERSOLD}, overbought={STOCH_OVERBOUGHT}")
print(f"Трейлинг-ТП: включён={TRAILING_TP}, отступ={TRAILING_TP_DISTANCE}%")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

price_log_scale_value = "log"
price_linear_scale_value = "linear"
price_log_scale_title = "Price (log scale)"
price_linear_scale_title = "Price (linear scale)"

fig_main.update_layout(
    title=f"🐵{discretization} | Entry={ENTRY_MODE} YellowEntry={USE_YELLOW_ENTRY} | ExitGreen={EXIT_MODE_FOR_GREEN}",
    xaxis_rangeslider_visible=False,
    yaxis1_type=price_log_scale_value if is_log_scale_by_default else price_linear_scale_value,
    yaxis1_title=price_log_scale_title if is_log_scale_by_default else price_linear_scale_title,
    height=1200,
    bargap=0,
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            active=1 if is_log_scale_by_default else 0,
            x=0.315,
            y=1.073,
            buttons=[
                dict(label="Linear", method="relayout", args=[{"yaxis.type": price_linear_scale_value, "yaxis.title.text": price_linear_scale_title}]),
                dict(label="Log", method="relayout", args=[{"yaxis.type": price_log_scale_value, "yaxis.title.text": price_log_scale_title}])
            ],
            font=dict(color="red", size=12, family="Arial"),
            bgcolor="rgba(0, 0, 0, 0)",
            bordercolor="red",
            borderwidth=1
        )
    ]
)

# ========== МАРКЕРЫ ==========
entries_long = df_backtest[df_backtest['signal'] == 1].index
entries_short = df_backtest[df_backtest['signal'] == -1].index

exits_tp = df_backtest[df_backtest['trade_type'].str.contains('tp', na=False) &
                       ~df_backtest['trade_type'].str.contains('trailing', na=False)].index

exits_tp_trailing = df_backtest[df_backtest['trade_type'].str.contains('tp_trailing', na=False)].index

exits_sl = df_backtest[df_backtest['trade_type'].str.contains('sl', na=False) &
                       ~df_backtest['trade_type'].str.contains('trailing', na=False)].index

fig_main.add_trace(go.Scatter(x=df_backtest.loc[entries_long, _KIEV_TIMESTAMP],
                               y=df_backtest.loc[entries_long, 'entry_price'],
                               mode='markers', name='Long Entry',
                               marker=dict(symbol='triangle-up', size=10, color='green')),
                   row=candlestick_row, col=1)

fig_main.add_trace(go.Scatter(x=df_backtest.loc[entries_short, _KIEV_TIMESTAMP],
                               y=df_backtest.loc[entries_short, 'entry_price'],
                               mode='markers', name='Short Entry',
                               marker=dict(symbol='triangle-down', size=10, color='red')),
                   row=candlestick_row, col=1)

fig_main.add_trace(go.Scatter(x=df_backtest.loc[exits_tp, _KIEV_TIMESTAMP],
                               y=df_backtest.loc[exits_tp, 'exit_price'],
                               mode='markers', name='Take Profit',
                               marker=dict(symbol='circle', size=10, color='lime',
                                           opacity=0.7, line=dict(width=1, color='darkgreen'))),
                   row=candlestick_row, col=1)

fig_main.add_trace(go.Scatter(x=df_backtest.loc[exits_tp_trailing, _KIEV_TIMESTAMP],
                               y=df_backtest.loc[exits_tp_trailing, 'exit_price'],
                               mode='markers', name='TP (Trailing)',
                               marker=dict(symbol='diamond', size=12, color='lightgreen',
                                           opacity=0.8, line=dict(width=1, color='green'))),
                   row=candlestick_row, col=1)

fig_main.add_trace(go.Scatter(x=df_backtest.loc[exits_sl, _KIEV_TIMESTAMP],
                               y=df_backtest.loc[exits_sl, 'exit_price'],
                               mode='markers', name='Stop Loss',
                               marker=dict(symbol='x', size=12, color='red',
                                           line=dict(width=2, color='darkred'))),
                   row=candlestick_row, col=1)

fig_main.show()
os.system('afplay /System/Library/Sounds/Glass.aiff')
print('\a')